In [1]:

import pandas as pd
import numpy as np
import re
import os

# Đọc VietJobs trực tiếp
url = "https://huggingface.co/datasets/dinhieufam/VietJobs/resolve/main/VietJobs.csv"

df = pd.read_csv(url, low_memory=False)

print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
display(df.head(3))


Số dòng: 48092
Số cột: 18


,job_title,location,country,qualifications,technical_skills,soft_skills,languages_required,experience_required,salary,contract_type,working_hours,benefits,description,requirements_text,category,salary_min,salary_max,salary_avg
0,Nhân Viên Vận Hành Hệ Thống Xử Lý Nước Thải,hồ chí minh,Việt Nam,"['Cao đẳng', 'Đại học chuyên ngành môi trường'...",NaN,"['Sức khoẻ tốt', 'Nhiệt tình', 'Chịu khó', 'Tr...",NaN,1 năm,15.0 - 22.0 triệu,Toàn thời gian,"Thứ 2 - Thứ 6 (từ 08:00 đến 17:00), Thứ 7 làm ...","['Thưởng năm', 'Lương tháng 13', 'Bảo hiểm y t...",- Vận hành hằng ngày các công trình được giao ...,"- Tốt nghiệp cao đẳng, đại học chuyên ngành mô...",nông_nghiệp_năng_lượng_môi_trường,15,22,18 triệu
1,Nhân Viên Vận Hành Hệ Thống Xử Lí Nước Thải,hồ chí minh,Việt Nam,"['Tốt nghiệp Trung cấp trở lên', 'Chuyên ngành...","['Vận hành hệ thống xử lý nước thải', 'Phân tí...","['Chăm chỉ', 'Có tinh thần trách nhiệm cao']",NaN,Không yêu cầu,8.0 - 13.0 triệu,Toàn thời gian,Làm việc theo ca,"['Chế độ phụ cấp công trình', 'Hưởng đầy đủ cá...","Vận hành, giám sát hệ thống xử lý nước thải th...",Giới tính: Nam. Độ tuổi: tuân thủ luật lao độn...,nông_nghiệp_năng_lượng_môi_trường,8,13,10 triệu
2,Nhân Viên Phát Triển Thị Trường,vĩnh long,Việt Nam,['Cao đẳng trở lên các chuyên ngành về Nông ng...,NaN,"['Tự tin giao tiếp', 'Diễn đạt thông tin tốt',...",NaN,Không yêu cầu,9.4 - 12.0 triệu,Toàn thời gian,"Thứ 2 - Thứ 6 (từ 07:30 đến 17:00), Thứ 7 (từ ...","['Phụ cấp điện thoại', 'Phụ cấp xăng xe', 'Thư...",Thực hiện quy trình thử nghiệm và đánh giá hạt...,Tốt nghiệp Cao đẳng trở lên các chuyên ngành v...,nông_nghiệp_năng_lượng_môi_trường,9,12,10 triệu


In [2]:

def quality_report(data):
    return pd.DataFrame({
        "column": data.columns,
        "dtype": data.dtypes.astype(str).values,
        "missing_count": data.isna().sum().values,
        "missing_percent": (
            data.isna().mean().values * 100
        ).round(2),
        "unique_count": [
            data[col].nunique(dropna=True)
            for col in data.columns
        ]
    })

quality_before = quality_report(df)

print("Dòng trùng hoàn toàn:", df.duplicated().sum())
display(quality_before)


Dòng trùng hoàn toàn: 385


,column,dtype,missing_count,missing_percent,unique_count
0,job_title,object,0,0.00,16720
1,location,object,0,0.00,984
2,country,object,0,0.00,10
3,qualifications,object,370,0.77,18167
4,technical_skills,object,6437,13.38,26426
5,soft_skills,object,2641,5.49,31790
6,languages_required,object,35796,74.43,123
7,experience_required,object,0,0.00,19
8,salary,object,0,0.00,839
9,contract_type,object,0,0.00,9


In [3]:

# Kiểm tra các quốc gia
print("===== COUNTRY =====")
print(df["country"].value_counts(dropna=False).to_string())

# Kiểm tra kinh nghiệm
print("\n===== EXPERIENCE =====")
print(df["experience_required"].value_counts(dropna=False).to_string())

# Kiểm tra mức lương
print("\n===== SALARY =====")
display(
    df[[
        "salary",
        "salary_min",
        "salary_max",
        "salary_avg"
    ]].head(20)
)

# Thống kê mức lương
print("\n===== SALARY STATISTICS =====")
display(df[["salary_min", "salary_max"]].describe())


===== COUNTRY =====
country
Việt Nam       47957
Mỹ                91
Nhật Bản          21
Lào                6
Đức                5
Campuchia          4
Mozambique         3
Philippines        3
Anh                1
Singapore          1

===== EXPERIENCE =====
experience_required
1 năm            15372
Không yêu cầu    12345
2 năm             9781
3 năm             5595
6 tháng           2258
5 năm             1934
4 năm              399
3 tháng            142
7 năm               91
10 năm              64
8 năm               34
6 năm               30
18 tháng            19
2 tháng             10
9 tháng              7
30 tháng             5
15 năm               3
12 năm               2
8 tháng              1

===== SALARY =====


,salary,salary_min,salary_max,salary_avg
0,15.0 - 22.0 triệu,15,22,18 triệu
1,8.0 - 13.0 triệu,8,13,10 triệu
2,9.4 - 12.0 triệu,9,12,10 triệu
3,12.0 - 15.0 triệu,12,15,13 triệu
4,15.0 - 25.0 triệu,15,25,20 triệu
5,50.0 triệu,50,50,50 triệu
6,10.0 - 15.0 triệu,10,15,12 triệu
7,10.0 - 20.0 triệu,10,20,15 triệu
8,12.0 - 20.0 triệu,12,20,16 triệu
9,0.0 triệu,0,0,Thoả thuận



===== SALARY STATISTICS =====


,salary_min,salary_max
count,48092.000000,48092.000000
mean,8.698037,13.533436
std,9.176023,16.779494
min,0.000000,0.000000
25%,0.000000,0.000000
50%,9.000000,12.000000
75%,12.000000,20.000000
max,500.000000,500.000000


# 1. Làm sạch

## Lọc VN, chuẩn hóa văn bản và xóa trùng

In [4]:

import pandas as pd
import numpy as np
import re
import unicodedata

jobs = df.copy()

# Lưu số dòng ban đầu
n_original = len(jobs)

# Chuẩn hóa khoảng trắng
for col in jobs.select_dtypes(include="object").columns:
    jobs[col] = jobs[col].apply(
        lambda x: " ".join(x.split())
        if isinstance(x, str) else x
    )

# Lọc Việt Nam
jobs = jobs[
    jobs["country"].str.strip().eq("Việt Nam")
].copy()

n_vietnam = len(jobs)

# Xóa các bản ghi trùng hoàn toàn
n_before_dedup = len(jobs)
jobs = jobs.drop_duplicates().reset_index(drop=True)
n_duplicates_removed = n_before_dedup - len(jobs)

print("Số dòng gốc:", n_original)
print("Số dòng thuộc Việt Nam:", n_vietnam)
print("Số dòng ngoài Việt Nam:", n_original - n_vietnam)
print("Số dòng trùng đã loại:", n_duplicates_removed)
print("Số dòng sau xử lý:", len(jobs))


Số dòng gốc: 48092
Số dòng thuộc Việt Nam: 47957
Số dòng ngoài Việt Nam: 135
Số dòng trùng đã loại: 393
Số dòng sau xử lý: 47564


Số dòng trùng đã loại ở đây là 393 vì chúng ta lọc Việt Nam trước, đồng thời chuẩn hóa khoảng trắng trước khi kiểm tra trùng.

## Chuẩn hóa kinh nghiệm thành số tháng

In [5]:

def experience_to_months(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text == "không yêu cầu":
        return 0

    match = re.fullmatch(
        r"(\d+)\s*(năm|tháng)", text
    )

    if not match:
        return np.nan

    number = int(match.group(1))
    unit = match.group(2)

    if unit == "năm":
        return number * 12

    return number


jobs["experience_months"] = (
    jobs["experience_required"]
    .apply(experience_to_months)
)

def experience_group(months):
    if pd.isna(months):
        return "Không rõ"
    if months == 0:
        return "Không yêu cầu"
    if months < 12:
        return "Dưới 1 năm"
    if months < 36:
        return "1-2 năm"
    if months < 60:
        return "3-4 năm"
    return "Từ 5 năm"


jobs["experience_group"] = (
    jobs["experience_months"]
    .apply(experience_group)
)

display(
    jobs[[
        "experience_required",
        "experience_months",
        "experience_group"
    ]].drop_duplicates().sort_values(
        "experience_months"
    )
)

print(
    "Không chuyển đổi được:",
    jobs["experience_months"].isna().sum()
)


,experience_required,experience_months,experience_group
1,Không yêu cầu,0,Không yêu cầu
4100,2 tháng,2,Dưới 1 năm
512,3 tháng,3,Dưới 1 năm
7,6 tháng,6,Dưới 1 năm
13087,8 tháng,8,Dưới 1 năm
4173,9 tháng,9,Dưới 1 năm
0,1 năm,12,1-2 năm
1957,18 tháng,18,1-2 năm
4,2 năm,24,1-2 năm
10447,30 tháng,30,1-2 năm


Không chuyển đổi được: 0


## Chuẩn hóa mức lương

In [6]:

def parse_salary(value):
    if pd.isna(value):
        return np.nan, np.nan, "Không rõ"

    text = str(value).strip().lower()
    text = text.replace(",", ".")

    # Các giá trị không có lương cụ thể
    if (
        "thoả thuận" in text
        or "thỏa thuận" in text
        or "thương lượng" in text
    ):
        return np.nan, np.nan, "Thỏa thuận"

    # Chỉ xử lý lương có đơn vị triệu
    if "triệu" not in text:
        return np.nan, np.nan, "Cần kiểm tra"

    # Lấy các giá trị số
    numbers = re.findall(r"\d+(?:\.\d+)?", text)

    if not numbers:
        return np.nan, np.nan, "Cần kiểm tra"

    values = [float(x) for x in numbers]

    # Lương bằng 0 được xem là chưa công bố
    if all(x == 0 for x in values):
        return np.nan, np.nan, "Thỏa thuận"

    # Khoảng lương
    if len(values) == 2:
        low, high = values

        if low > 0 and high >= low:
            return low, high, "Có khoảng lương"

    # Một mức lương duy nhất
    if len(values) == 1 and values[0] > 0:
        return values[0], values[0], "Một mức lương"

    return np.nan, np.nan, "Cần kiểm tra"


salary_result = jobs["salary"].apply(parse_salary)

jobs[[
    "salary_min_million",
    "salary_max_million",
    "salary_status"
]] = pd.DataFrame(
    salary_result.tolist(),
    index=jobs.index
)

# Trung điểm khoảng lương
jobs["salary_midpoint_million"] = (
    jobs["salary_min_million"]
    + jobs["salary_max_million"]
) / 2

print("Trạng thái mức lương:")
display(jobs["salary_status"].value_counts())

print("Thống kê mức lương đã xử lý:")
display(
    jobs[[
        "salary_min_million",
        "salary_max_million",
        "salary_midpoint_million"
    ]].describe()
)


Trạng thái mức lương:


,count
salary_status,
Có khoảng lương,31834
Thỏa thuận,13497
Một mức lương,2221
Cần kiểm tra,12


Thống kê mức lương đã xử lý:


,salary_min_million,salary_max_million,salary_midpoint_million
count,34055.000000,34055.000000,34055.000000
mean,12.176324,18.937356,15.556840
std,8.633171,17.068501,12.175806
min,0.100000,0.100000,0.100000
25%,8.000000,12.000000,10.500000
50%,10.000000,15.000000,13.500000
75%,15.000000,20.000000,17.500000
max,500.000000,500.000000,500.000000


## Các mức lương bất thường

In [7]:

# Đánh dấu khoảng lương cao bất thường
jobs["salary_outlier_flag"] = (
    jobs["salary_max_million"] > 100
)

print(
    "Số tin có lương tối đa trên 100 triệu:",
    jobs["salary_outlier_flag"].sum()
)

display(
    jobs.loc[
        jobs["salary_outlier_flag"],
        [
            "job_title",
            "category",
            "salary",
            "salary_min_million",
            "salary_max_million"
        ]
    ].head(30)
)

# Các chuỗi lương chưa xử lý được
print("Các định dạng lương cần kiểm tra:")

display(
    jobs.loc[
        jobs["salary_status"] == "Cần kiểm tra",
        "salary"
    ].value_counts().head(30)
)


Số tin có lương tối đa trên 100 triệu: 87


,job_title,category,salary,salary_min_million,salary_max_million
1600,Nhân Viên Kinh Doanh Bất Động Sản,marketing_truyền_thông_quảng_cáo_nội_dung,80.0 - 200.0 triệu,80.0,200.0
1721,Nhân Viên Kinh Doanh/ Tư Vấn/ Sales BĐS,marketing_truyền_thông_quảng_cáo_nội_dung,50.0 - 200.0 triệu,50.0,200.0
2156,Nhân Viên Kinh Doanh/ Tư Vấn/ Sales BĐS,marketing_truyền_thông_quảng_cáo_nội_dung,50.0 - 200.0 triệu,50.0,200.0
3169,Chuyên Viên Kinh Doanh - 100% Data Khách Marke...,marketing_truyền_thông_quảng_cáo_nội_dung,5.0 - 300.0 triệu,5.0,300.0
4046,Nhân Viên Quản Lý Kênh Social & Influencer,marketing_truyền_thông_quảng_cáo_nội_dung,8.0 - 150.0 triệu,8.0,150.0
4206,Nhân Viên Kinh Doanh,marketing_truyền_thông_quảng_cáo_nội_dung,10.0 - 200.0 triệu,10.0,200.0
5145,Nhân Viên Kinh Doanh/ Tư Vấn/ Sales Bất Động Sản,marketing_truyền_thông_quảng_cáo_nội_dung,80.0 - 200.0 triệu,80.0,200.0
5240,Nhân Viên Kinh Doanh/ Tư Vấn/ Sales Bất Động Sản,marketing_truyền_thông_quảng_cáo_nội_dung,80.0 - 150.0 triệu,80.0,150.0
5374,Nhân Viên Marketing (Nữ),marketing_truyền_thông_quảng_cáo_nội_dung,500.0 triệu,500.0,500.0
9235,Giám đốc nhân sự,nhân_sự_hành_chính_pháp_chế_tư_vấn,130.0 - 150.0 triệu,130.0,150.0


Các định dạng lương cần kiểm tra:


,count
salary,
0.0 - 20.0 triệu,2
0.0 - 10.0 triệu,2
8.0 - 0.0 triệu,2
0.0 - 13.5 triệu,2
0.0 - 8.0 triệu,2
0.0 - 7.0 triệu,1
0.0 - 4.0 triệu,1


In [8]:

# Xử lý 12 tin có một đầu khoảng lương bằng 0
mask = jobs["salary_status"] == "Cần kiểm tra"

for idx in jobs.index[mask]:
    salary_text = str(jobs.at[idx, "salary"])

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        salary_text.replace(",", ".")
    )

    if len(numbers) != 2:
        continue

    low, high = map(float, numbers)

    if low == 0 and high > 0:
        jobs.at[idx, "salary_min_million"] = np.nan
        jobs.at[idx, "salary_max_million"] = high
        jobs.at[idx, "salary_status"] = "Chỉ có lương tối đa"

    elif low > 0 and high == 0:
        jobs.at[idx, "salary_min_million"] = low
        jobs.at[idx, "salary_max_million"] = np.nan
        jobs.at[idx, "salary_status"] = "Chỉ có lương tối thiểu"

# Chỉ tính trung điểm khi có đủ hai đầu khoảng lương
jobs["salary_midpoint_million"] = (
    jobs["salary_min_million"]
    + jobs["salary_max_million"]
) / 2

print("Trạng thái lương sau khi xử lý:")
display(jobs["salary_status"].value_counts())

print("Số tin còn cần kiểm tra:")
print((jobs["salary_status"] == "Cần kiểm tra").sum())


Trạng thái lương sau khi xử lý:


,count
salary_status,
Có khoảng lương,31834
Thỏa thuận,13497
Một mức lương,2221
Chỉ có lương tối đa,10
Chỉ có lương tối thiểu,2


Số tin còn cần kiểm tra:
0


## Xử lý 87 tin có mức lương cao

In [9]:

# Giữ nguyên toàn bộ tin tuyển dụng
# Chỉ đánh dấu để phục vụ phân tích

jobs["salary_outlier_flag"] = (
    jobs["salary_max_million"] > 100
)

# Có đủ khoảng lương để tính trung điểm
jobs["salary_usable"] = (
    jobs["salary_midpoint_million"].notna()
)

# Nhóm lương tham khảo, loại các tin có trần >100 triệu
jobs["salary_for_analysis"] = (
    jobs["salary_midpoint_million"].where(
        ~jobs["salary_outlier_flag"]
    )
)

print("Tổng số tin:", len(jobs))
print("Tin có trung điểm lương:", jobs["salary_usable"].sum())
print("Tin có lương tối đa >100 triệu:", jobs["salary_outlier_flag"].sum())
print("Tin có lương tham khảo sau loại ngoại lệ:",
      jobs["salary_for_analysis"].notna().sum())


Tổng số tin: 47564
Tin có trung điểm lương: 34055
Tin có lương tối đa >100 triệu: 87
Tin có lương tham khảo sau loại ngoại lệ: 33968


Cột salary_for_analysis chỉ là một lựa chọn để TV2 vẽ biểu đồ tham khảo; ngưỡng 100 triệu là quy tắc phân tích do nhóm đặt ra, không phải tiêu chuẩn xác định tin tuyển dụng sai. Khi báo cáo, nên so sánh cả kết quả có và không loại nhóm này.

## Chuẩn hóa địa điểm và nhóm nghề

In [12]:

print("===== TOP 40 ĐỊA ĐIỂM =====")
display(jobs["location"].value_counts().head(40))

print("\n===== 16 NHÓM NGHỀ =====")
display(jobs["category"].value_counts())

print("\n===== MẪU KỸ NĂNG =====")
display(
    jobs[[
        "technical_skills",
        "soft_skills"
    ]].dropna().sample(
        min(10, len(jobs.dropna(
            subset=["technical_skills", "soft_skills"]
        ))),
        random_state=42
    )
)


===== TOP 40 ĐỊA ĐIỂM =====


,count
location,
hà nội,17976
hồ chí minh,15197
hà đông,1166
bắc ninh,931
đà nẵng,923
hải phòng,908
"hà nội, hồ chí minh",660
hưng yên,648
đồng nai,549



===== 16 NHÓM NGHỀ =====


,count
category,
kinh_doanh_bán_hàng_chăm_sóc_khách_hàng,8209
sản_xuất_lao_động_phổ_thông_cơ_khí,6329
marketing_truyền_thông_quảng_cáo_nội_dung,5947
tài_chính_kế_toán_ngân_hàng_bảo_hiểm,5431
du_lịch_nhà_hàng_khách_sạn_dịch_vụ,4194
thiết_kế_nghệ_thuật_giải_trí_truyền_hình_báo_chí,3374
nhân_sự_hành_chính_pháp_chế_tư_vấn,3202
xây_dựng_kiến_trúc_bất_động_sản,2760
công_nghệ_thông_tin_kỹ_thuật_số,1900



===== MẪU KỸ NĂNG =====


,technical_skills,soft_skills
12273,['Sử dụng thành thạo phần mềm tin học văn phòn...,['Có khả năng xử lý số lượng lớn dữ liệu quan ...
33306,"['Tin học văn phòng', 'Thương Mại Điện Tử']","['Giao tiếp tốt', 'Chăm chỉ', 'Trung thực', 'T..."
38144,"['Microsoft Office (Word, Excel)', 'Microsoft ...","['Kỹ năng giao tiếp', 'Kỹ năng lên kế hoạch', ..."
38099,['Tin học văn phòng'],"['Kỹ năng giao tiếp', 'Tư duy cởi mở', 'Linh h..."
5793,"['Facebook Ads', 'Google Ads', 'TikTok Ads', '...","['Kỹ năng phân tích dữ liệu', 'Tối ưu ngân sác..."
40991,['Kỹ thuật chụp ảnh y tế thông thường (X-quang...,"['Khả năng xử lý tình huống khẩn cấp', 'Tinh t..."
14176,"['Tiếng Trung (nghe, nói, đọc, viết)', 'Tin họ...","['Giao tiếp', 'Sắp xếp công việc', 'Tính linh ..."
40245,"['Kỹ năng sử dụng máy tính', 'Excel']","['Quản lý đội nhóm', 'Quản lý thời gian', 'Chị..."
9407,"['Phần mềm kế toán', 'Tin học văn phòng']","['Kỹ năng lãnh đạo', 'Tổ chức công việc', 'Quả..."
16394,"['Quản Lý Sản Xuất', 'Quản Lý Nhân Sự', 'Kiểm ...","['Kỹ năng tổ chức', 'Giám sát', 'Xử lý tình hu..."


# Nhận xét
có 3 vấn đề cần xử lý:

**Địa điểm:** Có 984 giá trị khác nhau, bao gồm tỉnh/thành, quận/huyện và nhiều địa điểm trong cùng một tin. Ví dụ hà đông thuộc Hà Nội, thủ đức thuộc TP. Hồ Chí Minh. Không được chỉ lấy địa điểm đầu tiên vì sẽ bỏ sót nhu cầu tuyển dụng ở địa điểm thứ hai.



**Nhóm nghề:** 16 nhóm đã được phân loại sẵn. Chúng ta chỉ cần thêm tên hiển thị dễ đọc để TV2 làm biểu đồ, không cần phân loại lại.



**Kỹ năng:** Dữ liệu có dạng chuỗi biểu diễn danh sách Python, chẳng hạn ['Excel', 'CAD']. Cần chuyển thành danh sách thật bằng ast.literal_eval, không tách bằng dấu phẩy vì một kỹ năng có thể chứa dấu phẩy.

In [ ]:
## Chuẩn hóa địa điểm

In [13]:

import re
import unicodedata

def normalize_text(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text

# Ánh xạ những địa điểm đã xác định được
location_mapping = {
    "hà đông": "Hà Nội",
    "bắc từ liêm": "Hà Nội",
    "nam từ liêm": "Hà Nội",
    "cầu giấy": "Hà Nội",
    "đống đa": "Hà Nội",

    "thủ đức": "TP. Hồ Chí Minh",
    "quận 7": "TP. Hồ Chí Minh",
    "bình thạnh": "TP. Hồ Chí Minh",
    "nhà bè": "TP. Hồ Chí Minh",
    "củ chi": "TP. Hồ Chí Minh",
    "hóc môn": "TP. Hồ Chí Minh",

    "tp vinh": "Nghệ An",
    "nha trang": "Khánh Hòa",

    "hà nội": "Hà Nội",
    "hồ chí minh": "TP. Hồ Chí Minh",
    "đà nẵng": "Đà Nẵng",
    "hải phòng": "Hải Phòng",
    "cần thơ": "Cần Thơ"
}

def clean_location(value):
    if pd.isna(value):
        return []

    text = normalize_text(value)

    # Tách các địa điểm cách nhau bằng dấu phẩy
    parts = [x.strip() for x in text.split(",")]

    result = []

    for part in parts:
        if not part:
            continue

        location = location_mapping.get(
            part,
            part.title()
        )

        if location not in result:
            result.append(location)

    return result

jobs["location_list"] = jobs["location"].apply(clean_location)

# Cột văn bản để lọc trên Streamlit
jobs["location_clean"] = jobs["location_list"].apply(
    lambda x: " | ".join(x)
)

display(
    jobs[["location", "location_clean"]].head(20)
)


,location,location_clean
0,hồ chí minh,TP. Hồ Chí Minh
1,hồ chí minh,TP. Hồ Chí Minh
2,vĩnh long,Vĩnh Long
3,nghệ an,Nghệ An
4,"hà nội, hồ chí minh",Hà Nội | TP. Hồ Chí Minh
5,hà nội,Hà Nội
6,hà nội,Hà Nội
7,hồ chí minh,TP. Hồ Chí Minh
8,biên hoà,Biên Hoà
9,hồ chí minh,TP. Hồ Chí Minh


## Chuẩn hóa 16 nhóm nghề

In [14]:

category_mapping = {
    "kinh_doanh_bán_hàng_chăm_sóc_khách_hàng":
        "Kinh doanh - Bán hàng - CSKH",

    "sản_xuất_lao_động_phổ_thông_cơ_khí":
        "Sản xuất - Cơ khí - Lao động phổ thông",

    "marketing_truyền_thông_quảng_cáo_nội_dung":
        "Marketing - Truyền thông",

    "tài_chính_kế_toán_ngân_hàng_bảo_hiểm":
        "Tài chính - Kế toán - Ngân hàng",

    "du_lịch_nhà_hàng_khách_sạn_dịch_vụ":
        "Du lịch - Nhà hàng - Khách sạn",

    "thiết_kế_nghệ_thuật_giải_trí_truyền_hình_báo_chí":
        "Thiết kế - Nghệ thuật - Truyền thông",

    "nhân_sự_hành_chính_pháp_chế_tư_vấn":
        "Nhân sự - Hành chính - Pháp chế",

    "xây_dựng_kiến_trúc_bất_động_sản":
        "Xây dựng - Kiến trúc - Bất động sản",

    "công_nghệ_thông_tin_kỹ_thuật_số":
        "Công nghệ thông tin",

    "logistics_vận_tải_chuỗi_cung_ứng":
        "Logistics - Vận tải - Chuỗi cung ứng",

    "kỹ_thuật_điện_điện_tử_viễn_thông":
        "Điện - Điện tử - Viễn thông",

    "giáo_dục_đào_tạo_nghiên_cứu":
        "Giáo dục - Đào tạo - Nghiên cứu",

    "y_tế_dược_chăm_sóc_sức_khỏe_công_nghệ_sinh_học":
        "Y tế - Dược - Công nghệ sinh học",

    "ngôn_ngữ_dịch_thuật":
        "Ngôn ngữ - Dịch thuật",

    "nhóm_nghề_khác":
        "Nhóm nghề khác",

    "nông_nghiệp_năng_lượng_môi_trường":
        "Nông nghiệp - Năng lượng - Môi trường"
}

jobs["category_clean"] = (
    jobs["category"].map(category_mapping)
)

print("Số nhóm nghề chưa ánh xạ:",
      jobs["category_clean"].isna().sum())

display(jobs["category_clean"].value_counts())


Số nhóm nghề chưa ánh xạ: 0


,count
category_clean,
Kinh doanh - Bán hàng - CSKH,8209
Sản xuất - Cơ khí - Lao động phổ thông,6329
Marketing - Truyền thông,5947
Tài chính - Kế toán - Ngân hàng,5431
Du lịch - Nhà hàng - Khách sạn,4194
Thiết kế - Nghệ thuật - Truyền thông,3374
Nhân sự - Hành chính - Pháp chế,3202
Xây dựng - Kiến trúc - Bất động sản,2760
Công nghệ thông tin,1900


## Chuẩn hóa kĩ năng

In [15]:

import ast

def parse_skills(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return [
                str(skill).strip()
                for skill in parsed
                if str(skill).strip()
            ]
    except (ValueError, SyntaxError):
        pass

    return []

jobs["technical_skills_list"] = (
    jobs["technical_skills"].apply(parse_skills)
)

jobs["soft_skills_list"] = (
    jobs["soft_skills"].apply(parse_skills)
)

jobs["technical_skills_clean"] = (
    jobs["technical_skills_list"].apply(
        lambda x: " | ".join(x)
    )
)

jobs["soft_skills_clean"] = (
    jobs["soft_skills_list"].apply(
        lambda x: " | ".join(x)
    )
)

print("Tin có kỹ năng chuyên môn:",
      jobs["technical_skills_list"].apply(bool).sum())

print("Tin có kỹ năng mềm:",
      jobs["soft_skills_list"].apply(bool).sum())

display(
    jobs[[
        "technical_skills_clean",
        "soft_skills_clean"
    ]].head(10)
)


Tin có kỹ năng chuyên môn: 41072
Tin có kỹ năng mềm: 44795


,technical_skills_clean,soft_skills_clean
0,,Sức khoẻ tốt | Nhiệt tình | Chịu khó | Trung thực
1,Vận hành hệ thống xử lý nước thải | Phân tích ...,Chăm chỉ | Có tinh thần trách nhiệm cao
2,,Tự tin giao tiếp | Diễn đạt thông tin tốt | Có...
3,,Sức khỏe tốt | Lanh lợi | Hoạt bát | Chịu khó ...
4,HACCP | ISO 22000 | VietGAP | GLOBAL GAP,Khả năng đi công tác | Linh động trong việc áp...
5,Thành thạo MS Office | Sử dụng AutoCad cơ bản ...,Kỹ năng phân tích | Tổng hợp dữ liệu | Khả năn...
6,AutoCAD | MS Office | SketchUp | PVSyst | ETAP,Tinh thần trách nhiệm cao | Sẵn sàng đi công tác
7,AutoCAD | SketchUp (cơ bản) | PVSyst | MS Office,Kỹ năng giám sát | Làm việc nhóm | Xử lý vấn đề
8,,
9,,Kỹ năng giao tiếp | Tư vấn | Đào tạo


## Bảng kỹ năng

In [16]:

technical_table = jobs[
    ["category_clean", "technical_skills_list"]
].explode("technical_skills_list")

technical_table = technical_table.rename(
    columns={"technical_skills_list": "skill"}
)

technical_table["skill_type"] = "Kỹ năng chuyên môn"

soft_table = jobs[
    ["category_clean", "soft_skills_list"]
].explode("soft_skills_list")

soft_table = soft_table.rename(
    columns={"soft_skills_list": "skill"}
)

soft_table["skill_type"] = "Kỹ năng mềm"

skills_table = pd.concat(
    [technical_table, soft_table],
    ignore_index=True
)

skills_table = skills_table.dropna(subset=["skill"])

skills_table["skill"] = (
    skills_table["skill"].astype(str).str.strip()
)

skills_table = skills_table[
    skills_table["skill"] != ""
].copy()

print("Số lượt kỹ năng:", len(skills_table))

display(
    skills_table["skill"].value_counts().head(20)
)


Số lượt kỹ năng: 327514


,count
skill,
Kỹ năng giao tiếp,7823
Trung thực,7632
Cẩn thận,6556
Nhanh nhẹn,5020
Giao tiếp tốt,4262
Làm việc nhóm,3693
Chăm chỉ,3578
Nhiệt tình,3506
Năng động,3294


In [17]:

import os
from google.colab import files

os.makedirs("TV1_output", exist_ok=True)

jobs = jobs.reset_index(drop=True)

jobs["job_id"] = [
    f"JOB_{i:06d}"
    for i in range(1, len(jobs) + 1)
]

# Không xuất trực tiếp các cột chứa list Python
list_columns = [
    "location_list",
    "technical_skills_list",
    "soft_skills_list"
]

jobs_export = jobs.drop(
    columns=list_columns,
    errors="ignore"
)

jobs_export.to_csv(
    "TV1_output/jobs_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

skills_table.to_csv(
    "TV1_output/skills_table.csv",
    index=False,
    encoding="utf-8-sig"
)

# Bảng đếm tin theo nhóm nghề
category_summary = (
    jobs.groupby("category_clean")
    .size()
    .reset_index(name="job_count")
    .sort_values("job_count", ascending=False)
)

category_summary.to_csv(
    "TV1_output/category_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã tạo các file:")
print(os.listdir("TV1_output"))


Đã tạo các file:
['jobs_clean.csv', 'skills_table.csv', 'category_summary.csv']


In [18]:

quality_comparison = pd.DataFrame({
    "Chỉ tiêu": [
        "Tổng số tin tuyển dụng",
        "Số tin ngoài Việt Nam",
        "Số dòng trùng hoàn toàn",
        "Số nhóm nghề",
        "Số tin thiếu kỹ năng chuyên môn",
        "Số tin thiếu kỹ năng mềm",
        "Số tin chưa có lương dạng số"
    ],
    "Trước xử lý": [
        len(df),
        (df["country"] != "Việt Nam").sum(),
        df.duplicated().sum(),
        df["category"].nunique(),
        df["technical_skills"].isna().sum(),
        df["soft_skills"].isna().sum(),
        np.nan
    ],
    "Sau xử lý": [
        len(jobs),
        (jobs["country"] != "Việt Nam").sum(),
        jobs.duplicated(
            subset=df.columns.tolist()
        ).sum(),
        jobs["category_clean"].nunique(),
        jobs["technical_skills_list"].apply(
            lambda x: len(x) == 0
        ).sum(),
        jobs["soft_skills_list"].apply(
            lambda x: len(x) == 0
        ).sum(),
        jobs["salary_midpoint_million"].isna().sum()
    ]
})

display(quality_comparison)

quality_comparison.to_csv(
    "TV1_output/quality_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)


,Chỉ tiêu,Trước xử lý,Sau xử lý
0,Tổng số tin tuyển dụng,48092.0,47564
1,Số tin ngoài Việt Nam,135.0,0
2,Số dòng trùng hoàn toàn,385.0,0
3,Số nhóm nghề,16.0,16
4,Số tin thiếu kỹ năng chuyên môn,6437.0,6492
5,Số tin thiếu kỹ năng mềm,2641.0,2769
6,Số tin chưa có lương dạng số,NaN,13509


**Tin thiếu kỹ năng lại tăng do:**

Trước xử lý: chỉ đếm NaN.



Sau xử lý: đếm danh sách kỹ năng rỗng [], bao gồm cả chuỗi rỗng, danh sách rỗng gốc và các giá trị không phân tích được.